<p><font size="6" color='grey'> <b>
Generative KI
</b></font> </br></p>
<p><font size="5" color='grey'> <b>
Retrieval-Augmented Generation (RAG) Chatbot für SQL-Datenbanken

---



# **1 <font color='orange'>|</font> Setup und Installation**
---



Zunächst erstellen wir eine requirements.txt mit allen benötigten Paketen:

In [1]:
%%writefile requirements.txt
langchain>=0.2.0
langchain-experimental>=0.0.49
langchain-openai>=0.0.5
openai==1.55.3
httpx==0.27.2
sqlalchemy>=2.0.0
gradio>=3.50.0
pydantic>=2.0.0

Writing requirements.txt


Installation der Pakete:

In [2]:
!pip install -q -U -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.0/209.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.2/320.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB

Module importieren:

In [3]:
# Standard Libraries
import os
import sqlite3
from typing import Any

# Third Party Libraries
import gradio as gr
from google.colab import userdata
from langchain_experimental.sql.base import SQLDatabase, SQLDatabaseChain
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

# **2 <font color='orange'>|</font> Konfiguration und Umgebungsvariablen**

In [4]:
# OpenAI API Konfiguration
OPENAI_API_KEY = userdata.get('OpenAI-API-Key')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# LLM Konfiguration
MODEL_NAME = "gpt-3.5-turbo"
TEMPERATURE = 0

# Datenbank Konfiguration
DB_PATH = "/content/sakila.db"
DB_URI = f"sqlite:///{DB_PATH}"

# App Konfiguration
APP_TITLE = "📚 Datenbank ChatGPT Chatbot"
APP_DESCRIPTION = "\n\n*Der Chatbot wertet die Datenbank Dokumente aus und beantwortet Fragen zum Inhalt*"

# LLM initialisieren
llm = ChatOpenAI(temperature=TEMPERATURE, model=MODEL_NAME)

# **3 <font color='orange'>|</font> Datenvorverarbeitung und Embedding**
---


Daten laden

In [17]:
!curl -L https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02%20data/nordwind.db -o nordwind.db

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0    101      0 --:--:-- --:--:-- --:--:--   101




# **4 <font color='orange'>|</font> Memory und Kontextmanagement**
---



Datenbankkontext laden

In [ ]:
def get_table_info(db_path: str) -> dict:
    """Extrahiert Schema-Informationen aus der SQLite Datenbank.

    Args:
        db_path: Pfad zur SQLite Datenbank

    Returns:
        Dictionary mit Tabellen und deren Spalten
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Tabellennamen abfragen
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [table[0] for table in cursor.fetchall()]

    # Spalten für jede Tabelle abfragen
    table_info = {}
    for table_name in tables:
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = [column[1] for column in cursor.fetchall()]
        table_info[table_name] = columns

    conn.close()
    return table_info

# Schema-Informationen laden
table_info = get_table_info(DB_PATH)

# **5 <font color='orange'>|</font> Prompt-Engineering**
---



SQLDatabaseChain Setup

In [7]:
# Workaround für Callback-Fehler
class Callbacks(BaseModel):
    pass
BaseCache = Callbacks
SQLDatabaseChain.model_rebuild()

# Datenbank-Chain initialisieren
db = SQLDatabase.from_uri(DB_URI)
db_chain = SQLDatabaseChain(llm=llm, database=db, verbose=False)

/usr/local/lib/python3.10/dist-packages/langchain_experimental/sql/base.py:77: UserWarning: Directly instantiating an SQLDatabaseChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  warnings.warn(


# **6 <font color='orange'>|</font> RAG-Chatbot-Architektur**

In [8]:
def run_query_with_schema_check(query: str,
                              db_chain=db_chain,
                              db_path=DB_PATH,
                              table_info=table_info) -> str:
    """Führt eine Benutzeranfrage unter Berücksichtigung des DB-Schemas aus.

    Args:
        query: Natürlichsprachliche Benutzeranfrage
        db_chain: LangChain SQLDatabaseChain
        db_path: Pfad zur Datenbank
        table_info: Schema-Informationen

    Returns:
        Antwort als String
    """
    # Schema-Beschreibung erstellen
    schema_info_str = "**Database Schema:**\n"
    for table_name, columns in table_info.items():
        schema_info_str += f"- **Table: {table_name}**\n"
        for column in columns:
            schema_info_str += f"  - {column}\n"

    # Prompt mit Schema anreichern
    modified_prompt = f"{schema_info_str}\n\n**User Query:** {query}"

    try:
        response = db_chain.invoke(modified_prompt)
        return f"{response['result']}"
    except sqlite3.OperationalError as e:
        if "no such table" in str(e):
            return f"Error: Die referenzierte Tabelle existiert nicht. {schema_info_str}"
        elif "no such column" in str(e):
            return f"Error: Die referenzierte Spalte existiert nicht. {schema_info_str}"
        else:
            return f"Fehler bei der Ausführung: {e}"

In [9]:
def call_chatbot(query: str, chat_history=[]) -> str:
    """Callback-Funktion für die Gradio-Schnittstelle.

    Args:
        query: Benutzeranfrage
        chat_history: Chat-Verlauf (nicht verwendet)

    Returns:
        Antwort des Chatbots
    """
    return run_query_with_schema_check(query)

In [ ]:
call_chatbot("", [])

# **7 <font color='orange'>|</font> Interaktions- und Testfunktionen**
---

Gradio Interface

In [10]:
# Beispielfragen definieren
EXAMPLE_QUESTIONS = [
    "Welche Kunden kommen aus Deutschland?",
    "Welche Produkte sind derzeit nicht verfügbar?",
    "Welche Regionen generieren den höchsten Umsatz?"
]

# Gradio Interface erstellen
demo = gr.ChatInterface(
    fn=call_chatbot,
    title=APP_TITLE,
    description=APP_DESCRIPTION,
    examples=EXAMPLE_QUESTIONS
)

/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:243: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Gradio App starten

In [11]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d51edc158be95aaccc.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **8 <font color='orange'>|</font> Erweiterte Funktionalitäten**
---



*Dieser Abschnitt ist für den einfachen SQL-Chatbot nicht relevant.*



# **9 <font color='orange'>|</font> Evaluation und Optimierung**
---

*Dieser Abschnitt ist für den einfachen SQL-Chatbot nicht relevant.*